# 13. Vision Transformer and detection blocks — Swin-T, FPN, CenterNet

Only tensor widths and input image size are reduced. Swin's resolution-dependent window/shift rule and DropPath path are preserved.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")


## 1. Swin-T with resolution-aware window and shift


In [ ]:
class DropPath(nn.Module):
    def __init__(self, probability=0.0):
        super().__init__()
        self.probability = probability

    def forward(self, x):
        if self.probability == 0.0 or not self.training:
            return x
        keep = 1.0 - self.probability
        shape = (x.size(0),) + (1,) * (x.ndim - 1)
        mask = (keep + torch.rand(shape, device=x.device, dtype=x.dtype)).floor()
        return x * mask / keep


def partition_windows(x, window):
    batch, height, width, channels = x.shape
    assert height % window == 0 and width % window == 0
    x = x.view(
        batch,
        height // window,
        window,
        width // window,
        window,
        channels,
    )
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window * window, channels)


def reverse_windows(windows, window, batch, height, width):
    channels = windows.size(-1)
    x = windows.view(
        batch,
        height // window,
        width // window,
        window,
        window,
        channels,
    )
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(batch, height, width, channels)


def relative_position_index(window, device):
    coords = torch.stack(
        torch.meshgrid(
            torch.arange(window, device=device),
            torch.arange(window, device=device),
            indexing="ij",
        )
    ).flatten(1)
    relative = coords[:, :, None] - coords[:, None, :]
    relative = relative.permute(1, 2, 0).contiguous()
    relative[:, :, 0] += window - 1
    relative[:, :, 1] += window - 1
    relative[:, :, 0] *= 2 * window - 1
    return relative.sum(-1)


def shifted_mask(height, width, window, shift, device):
    if shift == 0:
        return None
    region = torch.zeros(1, height, width, 1, device=device)
    h_slices = (
        slice(0, -window),
        slice(-window, -shift),
        slice(-shift, None),
    )
    w_slices = h_slices
    region_id = 0
    for h_slice in h_slices:
        for w_slice in w_slices:
            region[:, h_slice, w_slice] = region_id
            region_id += 1
    windows = partition_windows(region, window).squeeze(-1)
    diff = windows[:, None, :] - windows[:, :, None]
    return diff == 0


class SwinAttention(nn.Module):
    def __init__(self, dim, heads, input_resolution, target_window=7, target_shift=0):
        super().__init__()
        min_resolution = min(input_resolution)
        if min_resolution <= target_window:
            self.window = min_resolution
            self.shift = 0
        else:
            self.window = target_window
            self.shift = target_shift

        assert self.window > 0
        assert 0 <= self.shift < self.window
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.relative_bias = nn.Parameter(
            torch.zeros((2 * self.window - 1) ** 2, heads)
        )

    def forward(self, x):
        batch, height, width, channels = x.shape
        assert height % self.window == 0 and width % self.window == 0
        if self.shift:
            x = torch.roll(x, shifts=(-self.shift, -self.shift), dims=(1, 2))

        windows = partition_windows(x, self.window)
        qkv = self.qkv(windows).view(
            windows.size(0),
            self.window * self.window,
            3,
            self.heads,
            self.head_dim,
        )
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        index = relative_position_index(self.window, x.device)
        bias = self.relative_bias[index.reshape(-1)]
        bias = bias.view(self.window ** 2, self.window ** 2, self.heads)
        scores = scores + bias.permute(2, 0, 1)[None]

        mask = shifted_mask(height, width, self.window, self.shift, x.device)
        if mask is not None:
            mask = mask.repeat(batch, 1, 1)
            scores = scores.masked_fill(~mask[:, None], torch.finfo(scores.dtype).min)

        attended = scores.softmax(-1) @ v
        attended = attended.transpose(1, 2).contiguous().flatten(2)
        attended = self.out(attended)
        x = reverse_windows(attended, self.window, batch, height, width)
        if self.shift:
            x = torch.roll(x, shifts=(self.shift, self.shift), dims=(1, 2))
        return x


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, resolution, shifted, drop_path):
        super().__init__()
        target_shift = 3 if shifted else 0
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SwinAttention(dim, heads, resolution, 7, target_shift)
        self.drop_path1 = DropPath(drop_path)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )
        self.drop_path2 = DropPath(drop_path)

    def forward(self, x):
        x = x + self.drop_path1(self.attn(self.norm1(x)))
        return x + self.drop_path2(self.mlp(self.norm2(x)))


class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        _, height, width, _ = x.shape
        assert height % 2 == 0 and width % 2 == 0
        x00 = x[:, 0::2, 0::2]
        x10 = x[:, 1::2, 0::2]
        x01 = x[:, 0::2, 1::2]
        x11 = x[:, 1::2, 1::2]
        return self.reduction(self.norm(torch.cat([x00, x10, x01, x11], -1)))


class SmallWidthSwinTiny(nn.Module):
    def __init__(self, image_size=224, base_dim=24, classes=10, drop_path_rate=0.1):
        super().__init__()
        self.depths = [2, 2, 6, 2]
        self.heads = [3, 6, 12, 24]
        self.patch = nn.Conv2d(3, base_dim, 4, stride=4)
        self.patch_norm = nn.LayerNorm(base_dim)

        resolution = image_size // 4
        dims = [base_dim * (2 ** i) for i in range(4)]
        total_blocks = sum(self.depths)
        drop_rates = torch.linspace(0, drop_path_rate, total_blocks).tolist()
        drop_index = 0

        self.stages = nn.ModuleList()
        self.mergers = nn.ModuleList()
        for stage_index, (depth, heads, dim) in enumerate(zip(self.depths, self.heads, dims)):
            blocks = []
            for block_index in range(depth):
                blocks.append(
                    SwinBlock(
                        dim,
                        heads,
                        (resolution, resolution),
                        shifted=(block_index % 2 == 1),
                        drop_path=drop_rates[drop_index],
                    )
                )
                drop_index += 1
            self.stages.append(nn.ModuleList(blocks))
            if stage_index < 3:
                self.mergers.append(PatchMerging(dim))
                resolution //= 2

        self.norm = nn.LayerNorm(dims[-1])
        self.head = nn.Linear(dims[-1], classes)

    def forward(self, image):
        x = self.patch(image).permute(0, 2, 3, 1)
        x = self.patch_norm(x)
        for stage_index, blocks in enumerate(self.stages):
            for block in blocks:
                x = block(x)
            if stage_index < len(self.mergers):
                x = self.mergers[stage_index](x)
        return self.head(self.norm(x).mean((1, 2)))


swin = SmallWidthSwinTiny(image_size=224).to(device)
assert [len(stage) for stage in swin.stages] == [2, 2, 6, 2]
assert swin.stages[-1][0].attn.window == 7
assert swin.stages[-1][1].attn.shift == 0
swin(torch.randn(1, 3, 224, 224)).square().mean().backward()


## 2. Four-level FPN


In [ ]:
class FeaturePyramidNetwork(nn.Module):
    def __init__(self, input_channels=(16, 24, 32, 48), output_channels=12):
        super().__init__()
        self.lateral = nn.ModuleList(
            [nn.Conv2d(channels, output_channels, 1) for channels in input_channels]
        )
        self.smooth = nn.ModuleList(
            [nn.Conv2d(output_channels, output_channels, 3, padding=1) for _ in input_channels]
        )

    def forward(self, features):
        c2, c3, c4, c5 = features
        p5_inner = self.lateral[3](c5)
        p4_inner = self.lateral[2](c4) + F.interpolate(p5_inner, size=c4.shape[-2:], mode="nearest")
        p3_inner = self.lateral[1](c3) + F.interpolate(p4_inner, size=c3.shape[-2:], mode="nearest")
        p2_inner = self.lateral[0](c2) + F.interpolate(p3_inner, size=c2.shape[-2:], mode="nearest")
        p2 = self.smooth[0](p2_inner)
        p3 = self.smooth[1](p3_inner)
        p4 = self.smooth[2](p4_inner)
        p5 = self.smooth[3](p5_inner)
        p6 = F.max_pool2d(p5, 1, stride=2)
        return p2, p3, p4, p5, p6


fpn = FeaturePyramidNetwork()
features = (
    torch.randn(1, 16, 32, 32),
    torch.randn(1, 24, 16, 16),
    torch.randn(1, 32, 8, 8),
    torch.randn(1, 48, 4, 4),
)
assert len(fpn(features)) == 5


## 3. CenterNet heatmap focal loss, center regression, top-K decode


In [ ]:
class CenterNetHead(nn.Module):
    def __init__(self, channels=24, classes=3):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
        )
        self.heatmap = nn.Conv2d(channels, classes, 1)
        self.size = nn.Conv2d(channels, 2, 1)
        self.offset = nn.Conv2d(channels, 2, 1)
        nn.init.constant_(self.heatmap.bias, -2.19)

    def forward(self, x):
        h = self.shared(x)
        return self.heatmap(h), self.size(h), self.offset(h)


def centernet_focal(logits, target, alpha=2.0, beta=4.0):
    p = logits.sigmoid().clamp(1e-6, 1 - 1e-6)
    positive = target.eq(1).float()
    negative = target.lt(1).float()
    negative_weight = (1 - target).pow(beta)
    positive_loss = -(1 - p).pow(alpha) * p.log() * positive
    negative_loss = -p.pow(alpha) * (1 - p).log() * negative_weight * negative
    count = positive.sum().clamp_min(1.0)
    return (positive_loss.sum() + negative_loss.sum()) / count


def gather_centers(map_tensor, indices):
    batch, channels, height, width = map_tensor.shape
    flat = map_tensor.view(batch, channels, height * width).transpose(1, 2)
    expanded = indices[..., None].expand(-1, -1, channels)
    return flat.gather(1, expanded)


def topk_decode(heatmap, size, offset, k=10):
    scores = heatmap.sigmoid()
    batch, classes, height, width = scores.shape
    scores = F.max_pool2d(scores, 3, stride=1, padding=1).eq(scores) * scores
    flat_scores = scores.view(batch, -1)
    top_scores, top_indices = flat_scores.topk(k)
    class_ids = top_indices // (height * width)
    spatial = top_indices % (height * width)
    y = spatial // width
    x = spatial % width
    gathered_size = gather_centers(size, spatial)
    gathered_offset = gather_centers(offset, spatial)
    centers = torch.stack([x, y], -1).float() + gathered_offset
    return top_scores, class_ids, centers, gathered_size


head = CenterNetHead()
feature = torch.randn(1, 24, 16, 16)
heatmap, size, offset = head(feature)
target = torch.zeros_like(heatmap)
target[:, 0, 5, 6] = 1.0
loss = centernet_focal(heatmap, target)
indices = torch.tensor([[5 * 16 + 6]])
size_at_center = gather_centers(size, indices).abs().mean()
offset_at_center = gather_centers(offset, indices).abs().mean()
reg = size_at_center + offset_at_center
(loss + reg).backward()
assert len(topk_decode(heatmap, size, offset, k=5)) == 4


## Audit result

The reduced input no longer changes Swin's algorithm: resolution-dependent window shrinking, shift disabling, strict even PatchMerging, and DropPath are all explicit.
